## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

## Data Load and Preprocessing

In [ ]:
df = pd.read_csv("hungarian_heart_diseases.csv")
X = df.drop("outcome", axis=1)
y = df["outcome"]

## 1

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)
min_samples_leaves = [1, 3, 5, 10, 25, 50, 100]
train_accs = []
test_accs = []

for min_leaf in min_samples_leaves:
    dt = DecisionTreeClassifier(min_samples_leaf=min_leaf, random_state=1)
    dt.fit(X_train, y_train)
    
    y_train_pred = dt.predict(X_train)
    y_test_pred = dt.predict(X_test)
    train_accs.append(accuracy_score(y_train, y_train_pred))
    test_accs.append(accuracy_score(y_test, y_test_pred))

plt.plot(min_samples_leaves, train_accs, "-o", label="Training Accuracy")
plt.plot(min_samples_leaves, test_accs, "-s", label="Test Accuracy")
plt.xlabel("Min Samples Leaf")
plt.ylabel("Accuracy")
plt.title("Performance vs Min Number of Samples per Leaf")
plt.legend()
plt.show()

## 3

In [ ]:
# Train and temporary set split
X_train_fo, X_temp_fo, y_train_fo, y_temp_fo = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=1
)

# Validaion and test set split
X_val_fo, X_test_fo, y_val_fo, y_test_fo = train_test_split(
    X_temp_fo, y_temp_fo, test_size=0.5, stratify=y_temp_fo, random_state=1
)

best_val_acc = 0
best_max_depth, best_min_split = None, None
best_dt_model = None
max_depths = list(range(2, 5))
min_samples_splits = list(range(2, 101))

for max_depth in max_depths:
    for min_split in min_samples_splits:
        dt = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_split, random_state=1)
        dt.fit(X_train_fo, y_train_fo)
        
        y_val_pred_fo = dt.predict(X_val_fo)
        acc = accuracy_score(y_val_fo, y_val_pred_fo)
        
        if acc > best_val_acc:
            best_val_acc = acc
            best_max_depth = max_depth
            best_min_split = min_split
            best_dt_model = dt

y_test_pred_fo = best_dt_model.predict(X_test_fo)
test_acc = accuracy_score(y_test_fo, y_test_pred_fo)
print("Finished")

In [ ]:
print(f"""
Test Accuracy: {test_acc}
Validation Accuracy: {best_val_acc}
Max Depth: {best_max_depth}
Min Split: {best_min_split}
""")

plt.figure(figsize=(35, 18))
plot_tree(best_dt_model, 
          feature_names=X.columns,
          class_names=["normal", "heart disease"],
          filled=True,
          rounded=True,
          fontsize=16)
plt.title("Decision Tree Visualization")
plt.show()